### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.postprocessing.regex_mapping import RegexMapping
from quinex_application.visualisation.quinex_visualisation import QuinexVisualisation
from quinex_application.quinex_processing.quinex_postprocessing import QuinexPostprocessing

In [2]:
discussion = RegexMapping("../data/visualisation_files/regex/entity_patterns.json", "../data/visualisation_files/regex/property_patterns.json")
visualisation = QuinexVisualisation()
postprocessing = QuinexPostprocessing()

### **Load Results**

In [3]:
df_quinex_metadata = discussion.load_df("../data/quinex/df_quinex_metadata.parquet")
df_quinex = discussion.load_df("../data/quinex/df_quinex_adapted_new.parquet")

📂 Loaded DataFrame from '../data/quinex/df_quinex_metadata.parquet' with 31653 rows
📂 Loaded DataFrame from '../data/quinex/df_quinex_adapted_new.parquet' with 3244165 rows


### **Filter data**

In [ ]:
df_quinex[df_quinex["entity"].str.contains("mhi vestas", case=False)]["entity"].value_counts()

In [4]:
df = df_quinex.copy()
entity_regex_map = discussion.entity_regex_map
property_regex_map = discussion.property_regex_map

In [ ]:
property_regex_map = { 
    "Cost": {
        "Capital Cost": property_regex_map["Cost"]["Capital Cost"] 
    }
}
property_regex_map

In [ ]:
entity_regex_map = {
    "Wind": {
        "Onshore Wind": entity_regex_map["Wind"]["Onshore Wind"]
    }
}
entity_regex_map

In [5]:
df["entity_mapping"] = df["entity"].apply(lambda x: discussion.map_entity(x, entity_regex_map))
label_to_base = discussion.create_label_to_base(entity_regex_map)
df["entity_base_mapping"] = df["entity_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["entity_mapping"].notna()]

In [6]:
df["property_mapping"] = df["property"].apply(lambda x: discussion.map_entity(x, property_regex_map))
label_to_base = discussion.create_label_to_base(property_regex_map)
df["property_base_mapping"] = df["property_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["property_mapping"].notna()]

In [ ]:
df["property_mapping"].value_counts()

In [ ]:
df[df["property_mapping"] == "Power Density"]["entity_mapping"].value_counts()

In [7]:
df.to_parquet("../data/visualisation_files/df_technologies.parquet")

### **Postprocess Units**

In [8]:
import pandas as pd
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
technologies_df = technologies_df[technologies_df["property_mapping"] == "Capital Cost"]
def dicts_to_tuples(units):
    if units is None:
        return None
    return [(u.get('text', ''), u.get('exponent', None), u.get('unit_uri', ''), u.get('year', None)) 
            for u in units]


technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(dicts_to_tuples)

In [9]:
import json

technologies_df = technologies_df.rename(columns={
    "value_normalized": "value_normalized_old",
    "unit_normalized": "unit_normalized_old",
    "unit_dimension": "unit_dimension_old",
    "value_normalized_adapted": "value_normalized",
    "unit_normalized_adapted": "unit_normalized",
    "unit_dimension_adapted": "unit_dimension"
})

# Deserialisiere unit_normalized (war JSON-String in Parquet)
technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)


technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(dicts_to_tuples)

In [10]:
# CAPEX Dimension
technologies_df = technologies_df[(technologies_df["unit_dimension"].isin([
    "A0E0L-2I0M-1H0T3D0", 
    #"A0E0L0I0M0H0T0D1"
    ]))]

# LCOE Dimension
# technologies_df = technologies_df[(technologies_df["unit_dimension"].isin(["A0E0L-2I0M-1H0T2D0"]))]

In [11]:
def normalize_compound_units(units):
    """
    Zerlegt zusammengesetzte Units (z.B. €/kW) in ihre Komponenten.
    """
    if units is None:
        return None
    
    result = []
    
    for unit_tuple in units:
        text, exponent, uri, year = unit_tuple
        
        # Prüfe auf EUR-PER-KiloW Muster
        if "EUR-PER-KiloW" in uri:
            result.append(("€", 1.0, "http://qudt.org/vocab/unit/CCY_EUR", year))
            result.append(("kW", -1.0, "http://qudt.org/vocab/unit/KiloW", year))
        # Prüfe auf EUR-PER-W Muster
        elif "EUR-PER-W" in uri:
            result.append(("€", 1.0, "http://qudt.org/vocab/unit/CCY_EUR", year))
            result.append(("W", -1.0, "http://qudt.org/vocab/unit/W", year))
        else:
            # Nicht zusammengesetzt → behalte original
            result.append(unit_tuple)
    
    return result

# Wende die Normalisierung an
technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(normalize_compound_units)

In [12]:
print(technologies_df["unit_normalized"].value_counts())
print("--------------------------------------------------------------------------------")
exclude_values = [
    [(None, None, 'http://qudt.org/vocab/unit/UNITLESS', None)],
    [('%', 1.0, 'http://qudt.org/vocab/unit/PERCENT', None)], 
]
technologies_df = technologies_df[~technologies_df["unit_normalized"].isin(exclude_values)]
print(technologies_df["unit_normalized"].value_counts().head(20))

unit_normalized
[(€, 1.0, http://qudt.org/vocab/unit/CCY_EUR, None), (kW, -1.0, http://qudt.org/vocab/unit/KiloW, None)]     249
[($, 1, http://qudt.org/vocab/unit/CCY_USD, None), (/kW, -1, http://qudt.org/vocab/unit/KiloW, None)]        200
[($, 1, http://qudt.org/vocab/unit/CCY_USD, None), (kW, -1, http://qudt.org/vocab/unit/KiloW, None)]         196
[(USD, 1, http://qudt.org/vocab/unit/CCY_USD, None), (kW, -1, http://qudt.org/vocab/unit/KiloW, None)]       111
[($, 1, http://qudt.org/vocab/unit/CCY_USD, None), (per kW, -1, http://qudt.org/vocab/unit/KiloW, None)]      46
                                                                                                            ... 
[(EUR, 1, http://qudt.org/vocab/unit/CCY_EUR, None), (MW, -1, http://qudt.org/vocab/unit/MegaW, None)]         1
[(INR, 1, http://qudt.org/vocab/unit/CCY_INR, None), (/MW, -1, http://qudt.org/vocab/unit/MegaW, None)]        1
[(EUR, 1, http://qudt.org/vocab/unit/CCY_EUR, None), (/kW, -1, http://qudt.org/v

In [13]:
# --- 1. Erste Normalisierung: USD pro Watt (oder GW) ---
# target_unit_full = [("$", 1.0, "http://qudt.org/vocab/currency/USD", None),
#                     ("kWh", -1.0, "http://qudt.org/vocab/unit/KiloW-HR", None)]
target_unit_full = [("$", 1.0, "http://qudt.org/vocab/unit/CCY_USD", None),
                    ("W", -1.0, "http://qudt.org/vocab/unit/W", None)]

technologies_df['value_converted'] = technologies_df.apply(
    lambda row: visualisation.convert_row(
        row, 
        target_unit_full,
        #2024
      ),
    axis=1
)

c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Conversion error for 2146: Could not get IMF data point for indicator ER.G163.USD_XDC.PA_RT.M for year 1990.


c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Conversion error for 2147: Could not get IMF data point for indicator ER.G163.USD_XDC.PA_RT.M for year 1994.


c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Conversion error for 2184: Could not get IMF data point for indicator ER.G163.USD_XDC.PA_RT.M for year 1994.
Conversion from 2005 to 2050 is not possible, because economic data in annual resolution is not available for the current year or future years.
Conversion from 2005 to 2035 is not possible, because economic data in annual resolution is not available for the current year or future years.
Conversion from 2030 to 2030 is not possible, because economic data in annual resolution is not available for the current year or future years.
Conversion from 2030 to 2030 is not possible, because economic data in annual resolution is not available for the current year or future years.
Conversion from 2050 to 2050 is not possible, because economic data in annual resolution is not available for the current year or future years.
Conversion from 2030 to 2030 is not possible, because economic data in annual resolution is not available for the current year or future years.
Conversion from 2030 to 203

In [ ]:
technologies_df.head(1)["temporal_scope"]

In [ ]:
# --- 2. Zweite Normalisierung: nur USD, für Zeilen mit None ---
# target_unit_usd = [("$", 1.0, "http://qudt.org/vocab/currency/USD", None)]

# mask_none = technologies_df['value_converted'].isna()

# technologies_df.loc[mask_none, 'value_converted'] = technologies_df.loc[mask_none].apply(
#     lambda row: visualisation.convert_row(row, target_unit_usd),
#     axis=1
# )

In [ ]:
# import math

# def scale_usd_per_watt(value):
#     if value is None or (isinstance(value, float) and math.isnan(value)):
#         return value
    
#     abs_val = abs(value)
    
#     if abs_val < 10:           # einstelliger Bereich → USD/W
#         return value
#     elif abs_val < 1e4:        # ca. 10–10.000 → USD/kW
#         return value / 1e3
#     elif abs_val < 1e7:        # 10.000 – 10.000.000 → USD/MW
#         return value / 1e6
#     else:                      # alles darüber hinaus → NaN
#         return float('nan')

# technologies_df['value_converted'] = technologies_df['value_converted'].apply(scale_usd_per_watt)

In [ ]:
df_test = technologies_df[technologies_df["entity_mapping"] == "Offshore Wind"]
df_test[["quantity_text", "temporal_scope", "value_normalized", "unit_normalized", "value_converted"]].iloc[30:60]

In [ ]:
technologies_df.tail(5)

In [ ]:
technologies_df["unit_normalized"].value_counts()

In [14]:
technologies_df = technologies_df[technologies_df["value_converted"].notna()]
technologies_df = technologies_df.reset_index(drop=True)
technologies_df.to_excel("../data/visualisation_files/unit_converted/df_technologies_capex_new.xlsx")

In [ ]:
technologies_df